## Whisper Model

## Pre-Processing

In [ ]:
!pip install datasets
!pip install git+https://github.com/huggingface/transformers
!pip install librosa
!pip install evaluate
!pip install jiwer
!pip install transformers[torch]  # additional requirements for training in colab
!pip install accelerate -U        # additional requirements for training in colab

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset
from datasets import Audio
import pandas as pd

# Load the Mozilla Common Voice dataset for Tamil
dataset = load_dataset("mozilla-foundation/common_voice_11_0", "ta")

dataset = dataset.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])

# Select 2000 rows for training and 400 rows for testing
train_dataset = dataset["train"].select(range(2000))
test_dataset = dataset["test"].select(range(400))

print(train_dataset)


# Convert the datasets to pandas DataFrames if needed
train_df = train_dataset.to_pandas()
test_df = test_dataset.to_pandas()

# Display the first few rows of the training dataset
train_df.head()


In [ ]:
# create arrays of each audio file and append those values as a column in the above datasets.
# To do this we will use the cast_column function from Dataset.
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

### Prepare Feature Extractor, Tokenizer and Data
### Load WhisperFeatureExtractor and Tokenizer

In [ ]:
# !pip install transformers

In [ ]:
# !pip install --upgrade datasets transformers accelerate soundfile librosa evaluate jiwer tensorboard gradio

In [ ]:
## import feature extractor and tokenizer
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer

feature_extractor = WhisperFeatureExtractor.from_pretrained("vasista22/whisper-tamil-medium")
tokenizer = WhisperTokenizer.from_pretrained("vasista22/whisper-tamil-medium",task="transcribe")

In [ ]:
## Combine To Create A WhisperProcessor

from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained("vasista22/whisper-tamil-medium", task="transcribe")

### Preparing data for the model
    * generating log-mel spectrogram
    * creating label_ids using tokenizer

In [ ]:
train_dataset[0]

In [ ]:
input_str = train_dataset[0]["sentence"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


In [ ]:
def prepare_dataset(examples):
    # compute log-Mel input features from input audio array
    audio = examples["audio"]
    examples["input_features"] = feature_extractor(
        audio["array"], sampling_rate=16000).input_features[0]

    del examples["audio"]

    sentences = examples["sentence"]

    # encode target text to label ids
    examples["labels"] = tokenizer(sentences).input_ids

    del examples["sentence"]
    return examples

#### Using the dataset’s.map method, we can apply the data preparation function to each of our training examples

In [ ]:
train_dataset = train_dataset.map(prepare_dataset, num_proc=2)
test_dataset = test_dataset.map(prepare_dataset, num_proc=2)

### Training and Evaluation
    * Defining a data-collator to handle input data and label spearetly
    * The word error rate (WER) metric will be used to measure the effectiveness of the model. A compute metrics function must be created to handle this calculation.
    * Loading a pre-trained checkpoint: A pre-trained checkpoint must be loaded and properly set up for training.
    * Defining the training arguments: The trainer will use these to create the training schedule.

In [ ]:
train_dataset

In [ ]:
## data-collator
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
## Evalution metric
import evaluate
metric = evaluate.load("wer")

In [ ]:
# evaluation metric function
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [ ]:
# Loading a Pre-Trained Checkpoint
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("vasista22/whisper-tamil-medium")

In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="REDACTED-M/KAGGLE-NOTEBOOK-WHISPER-1",  # change to a repo name of your choice
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=200,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
    optim="adamw_bnb_8bit"
)

In [ ]:
## training
from transformers import Seq2SeqTrainer
# early_stopping_callback = EarlyStoppingCallback(early_stopping_threshold=0.001, early_stopping_patience=2)
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    # callbacks=[early_stopping_callback]
)

In [ ]:
!pip install bitsandbytes

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained(training_args.output_dir)
processor.save_pretrained(training_args.output_dir)